# Bài 6 · Kết nối & truy xuất dữ liệu ngoài

**Lập trình xử lý dữ liệu · 2627-1 · Viện TTNT, UET-VNU**

Chọn **Save a copy in Drive** trước khi sửa.

## Mục tiêu bài học

1. Đọc file và giữ đúng mã, kiểu dữ liệu, giá trị thiếu.
2. Lấy dữ liệu qua API, hiểu cấu trúc JSON và đơn vị đo.
3. Dùng SQL để lọc, nhóm và ghép bảng; kiểm chứng kết quả trên dữ liệu nhỏ.

Các bảng nhỏ và JSON trên slide là **dữ liệu giả lập**. Ví dụ dữ liệu thật dùng bản CSV rút gọn của Santiago, snapshot **29/06/2026**; giá tính bằng **CLP/đêm**. Ví dụ gọi Open-Meteo lấy dự báo tại thời điểm chạy.

**Giáo trình:** [McKinney, chương 6](https://wesmckinney.com/book/accessing-data), §6.1–6.4. Bài dùng Parquet và DuckDB để thực hành lưu bảng và truy vấn SQL.

### Chuẩn bị

Ô cài đặt chạy trước khi import thư viện. Nếu phiên Colab đã nạp pandas phiên bản khác, chọn **Restart session and run all** sau khi cài.

In [ ]:
%pip install -q "pandas==3.0.3" "duckdb==1.5.5" "pyarrow==25.0.1" requests

In [ ]:
import json
import time
from io import StringIO
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.width", 100)
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
print("pandas:", pd.__version__)

## 1. Đọc và lưu file

CSV lưu nội dung dưới dạng văn bản. Khi đọc, pandas phải suy luận kiểu dữ liệu và nhận diện giá trị thiếu.

### CSV không lưu kiểu dữ liệu

File `cho_o.csv` dưới đây dùng dữ liệu giả lập, giá tính bằng USD/đêm. Mã chỗ ở có ba ký tự; `?` nghĩa là chưa có giá.

In [ ]:
csv_text = """id,khu,gia
001,A,100
002,B,80
003,A,200
004,B,120
005,A,?
"""
Path("data/raw/cho_o.csv").write_text(csv_text, encoding="utf-8")
print(csv_text)

In [ ]:
suy_luan = pd.read_csv("data/raw/cho_o.csv")
print(suy_luan.to_string(index=False))

Mã `001` bị đọc thành số `1`; cột `gia` vẫn là chuỗi vì có `?`. Kiểm tra kiểu từng cột:

In [ ]:
print(suy_luan.dtypes)

### Khai báo cách đọc

`dtype` quy định kiểu cột; `na_values` khai báo chuỗi nào là giá trị thiếu, theo từng cột. Mặc định pandas coi 19 chuỗi như `NA`, `null`, `N/A` và ô trống là thiếu; `keep_default_na=False` tắt danh sách này để chỉ dùng quy ước của file. Với file này, tham số chưa làm đổi kết quả; nó cần khi một giá trị hợp lệ trùng danh sách mặc định.

In [ ]:
df = pd.read_csv(
    "data/raw/cho_o.csv",
    dtype={"id": "str", "gia": "float64"},
    keep_default_na=False, na_values={"gia": ["?"]},
)
print(df.to_string(index=False))

In [ ]:
assert df["id"].tolist() == ["001", "002", "003", "004", "005"]
assert df["gia"].isna().sum() == 1
print(df.dtypes)

### Bài tập 1: Đọc bảng giá

File `gia_theo_khu.csv` dùng dữ liệu giả lập (USD/đêm):

```csv
khu,gia
NA,100
TB,80
NA,
TB,0
```

`NA` là mã khu hợp lệ; ô trống là chưa có giá; `0` là giá đã ghi nhận.

**Viết mã tính giá trung bình từng khu từ file này.**

In [ ]:
gia_theo_khu = """khu,gia
NA,100
TB,80
NA,
TB,0
"""
Path("data/raw/gia_theo_khu.csv").write_text(gia_theo_khu, encoding="utf-8")
print(gia_theo_khu)

In [ ]:
# Bài tập 1: viết mã của bạn ở đây.

### Lưu bảng để dùng lại

Sau khi đã xác định đúng mã, kiểu cột và giá trị thiếu, ta cần giữ bảng này cho các lần phân tích sau. Nếu ghi CSV, lần đọc sau lại phải xác định kiểu từ văn bản hoặc khai báo kiểu.

**Parquet** là định dạng file nhị phân dành cho dữ liệu dạng bảng:

| Nhu cầu | Cách Parquet lưu dữ liệu |
|---|---|
| Giữ kiểu cột đã xác định | Lưu kiểu cùng dữ liệu |
| Chỉ đọc các cột cần dùng | Tổ chức dữ liệu theo cột |
| Giảm dung lượng lưu trữ | Nén dữ liệu theo từng cột |

Dữ liệu được chia thành các nhóm hàng; trong mỗi nhóm, các cột được lưu riêng. Bộ đọc có thể bỏ qua dữ liệu của các cột không dùng; với CSV, `usecols` vẫn phải quét cả dòng văn bản.

CSV vẫn tiện để xem và trao đổi văn bản. Parquet phù hợp cho bảng đã xử lý được đọc lại nhiều lần; kích thước và tốc độ còn phụ thuộc dữ liệu, cách nén và thao tác đọc. Phần tự học có mã để tự đo trên cùng một bảng.

### Lưu và đọc lại bằng pandas

Lưu `df` đã đọc đúng ở đầu mục, rồi đọc lại và kiểm tra mã cùng kiểu cột. Giữ file tải về trong `data/raw/`; ghi bảng xử lý vào `data/processed/`.

Nguồn: [Apache Parquet](https://parquet.apache.org/docs/overview/).

In [ ]:
df.to_parquet("data/processed/cho_o.parquet", index=False)
lai = pd.read_parquet("data/processed/cho_o.parquet")
print(lai.loc[0, "id"], lai["gia"].dtype)

In [ ]:
pd.testing.assert_frame_equal(df, lai)

### Áp dụng cho dữ liệu Santiago

Dùng [bản CSV rút gọn ngày 29/06/2026](https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/listings.csv): 18.534 hàng, 19 cột. Cột `price` trong **bản này** đã là số, đơn vị CLP/đêm.

Tải và giữ nguyên file gốc. Khi file đã có, dùng lại bản đã tải. Khi đọc, `id` là mã nên đọc thành chuỗi, như `001` ở trên; `usecols` chỉ đọc bốn cột cần dùng. Bảng được lưu thành Parquet để dùng lại ở mục 3.

In [ ]:
URL_LISTINGS = (
    "https://data.insideairbnb.com/chile/rm/santiago/"
    "2026-06-29/visualisations/listings.csv"
)
tep_listings = Path("data/raw/santiago_2026-06-29.csv")
if not tep_listings.exists():
    response = requests.get(URL_LISTINGS, timeout=60)
    response.raise_for_status()
    tep_listings.write_bytes(response.content)

In [ ]:
listings = pd.read_csv(
    "data/raw/santiago_2026-06-29.csv",
    usecols=["id", "neighbourhood", "room_type", "price"],
    dtype={"id": "str"},
)
print(listings.shape)
listings.to_parquet("data/processed/listings.parquet", index=False)

In [ ]:
assert len(listings) == 18_534
assert pd.api.types.is_numeric_dtype(listings["price"])

## 2. Lấy dữ liệu qua API

### Tự động lấy dữ liệu từ dịch vụ

Dữ liệu nằm trên hệ thống của nhà cung cấp và thay đổi theo thời gian, như dự báo thời tiết được cập nhật nhiều lần mỗi ngày. Chương trình cần cập nhật tự động; mỗi lần chỉ cần một phần dữ liệu: một địa điểm, vài ngày tới, một mã sản phẩm.

Nhà cung cấp cần quy định cách chương trình gửi yêu cầu và nhận đúng phần dữ liệu đó: cách gọi, tham số hợp lệ và cấu trúc phản hồi. Những dữ liệu và điều kiện có thể yêu cầu phụ thuộc vào từng dịch vụ.

### Trao đổi dữ liệu qua API

**API** (giao diện lập trình ứng dụng, Application Programming Interface) quy định cách các chương trình tương tác với nhau. Với **web API**, chương trình gửi yêu cầu đến dịch vụ và nhận phản hồi qua HTTP, giao thức yêu cầu–phản hồi trên web. Chương trình gọi API có thể là ứng dụng web, ứng dụng di động hay script tự động.

- Chương trình gửi yêu cầu tới địa chỉ của dịch vụ, kèm các tham số cần thiết.
- Dịch vụ xử lý yêu cầu rồi trả phản hồi về chương trình, gồm mã trạng thái và nội dung.

Mạch xử lý trong phần này: xác định yêu cầu → gửi yêu cầu → đọc phản hồi → chọn phần JSON để tạo bảng → lưu lại dữ liệu đã nhận.

### Ví dụ: lấy dự báo thời tiết

Cần lấy nhiệt độ tối đa ba ngày tại Santiago, cập nhật theo thời điểm chạy. Ta dùng web API của dịch vụ Open-Meteo.

[Tài liệu Open-Meteo](https://open-meteo.com/en/docs) cho biết endpoint (địa chỉ nhận yêu cầu): `https://api.open-meteo.com/v1/forecast`.

| Yêu cầu | Tham số | Giá trị |
|---|---|---|
| Vị trí minh hoạ tại Santiago | `latitude`, `longitude` | `-33.45`, `-70.66` |
| Nhiệt độ tối đa mỗi ngày | `daily` | `temperature_2m_max` |
| Ba ngày, kể từ hôm nay | `forecast_days` | `3` |
| Ngày theo giờ địa phương | `timezone` | `America/Santiago` |

### Gửi yêu cầu bằng Python

Thư viện `requests` dùng `get()` để gửi yêu cầu lấy dữ liệu qua HTTP. `params` chứa các tham số trong bảng; thư viện tự ghép chúng vào URL. Có thể xem URL đã gửi bằng `r.url`.

`timeout=20` giới hạn thời gian chờ kết nối và chờ nhận dữ liệu; đây không phải giới hạn tổng thời gian tải. `r` là đối tượng phản hồi; `r.status_code` cho biết mã trạng thái HTTP. Mã `200` cho biết yêu cầu thành công.

Ô dưới gọi API thật. Chạy lại có thể nhận được bản dự báo mới.

In [ ]:
import requests

r = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": -33.45, "longitude": -70.66,
            "daily": "temperature_2m_max",
            "timezone": "America/Santiago", "forecast_days": 3},
    timeout=20,
)
print(r.status_code)

URL đầy đủ đã gửi. Mở URL này trên trình duyệt sẽ thấy đúng JSON mà chương trình nhận:

In [ ]:
print(r.url)

### Đọc dữ liệu trong phản hồi

Open-Meteo trả nội dung dạng **JSON** (JavaScript Object Notation): văn bản chứa các cặp tên–giá trị và mảng.

1. `r.raise_for_status()` báo lỗi nếu máy chủ trả mã HTTP lỗi, như 400 hoặc 500.
2. `r.json()` chuyển JSON sang đối tượng Python. Đối tượng JSON thành `dict`, mảng JSON thành `list`; trong ví dụ này, lớp ngoài là `dict` có 9 khoá, dữ liệu theo ngày nằm trong khoá `daily`.

Đọc được JSON chưa chứng tỏ yêu cầu thành công: nội dung báo lỗi cũng có thể là JSON. Vì vậy kiểm mã lỗi trước khi đọc dữ liệu.

In [ ]:
r.raise_for_status()
du_lieu = r.json()
print(type(du_lieu).__name__, len(du_lieu))

In [ ]:
print(list(du_lieu))

### Chọn phần JSON để tạo bảng

Dùng **phản hồi rút gọn với số liệu giả lập** dưới đây, chỉ giữ ba khoá cần dùng, để theo dõi ví dụ trên slide. Ngày và nhiệt độ cố định này không phải kết quả của lần gọi API thật ở trên. `mau` là `dict` Python sau khi đọc JSON.

- `daily` chứa hai danh sách khớp theo vị trí: phần tử thứ i của `time` ứng với phần tử thứ i của `temperature_2m_max`; mỗi vị trí là một ngày.
- `daily_units` cho biết đơn vị của từng đại lượng.
- `timezone` cho biết múi giờ dùng để xác định ngày.

Muốn mỗi hàng là một ngày, ta cần phần `daily`. Các thông tin đơn vị và múi giờ được đọc riêng.

In [ ]:
mau = {
    "timezone": "America/Santiago",
    "daily_units": {"temperature_2m_max": "°C"},
    "daily": {
        "time": ["2026-10-01", "2026-10-02", "2026-10-03"],
        "temperature_2m_max": [22.0, 26.0, 24.0],
    },
}

### Tạo bảng theo ngày

Chọn `daily` vì mỗi hàng cần ứng với một ngày. Metadata như đơn vị và múi giờ được đọc riêng để diễn giải các cột.

In [ ]:
thoi_tiet = pd.DataFrame(mau["daily"])
print(thoi_tiet.to_string(index=False))

### Áp dụng với phản hồi thật

Làm tương tự với `du_lieu` nhận từ API. Đọc đơn vị và múi giờ cùng với bảng; ngày và nhiệt độ sẽ thay đổi theo thời điểm gọi.

In [ ]:
print("Múi giờ:", du_lieu["timezone"])
print("Đơn vị:", du_lieu["daily_units"])
du_bao = pd.DataFrame(du_lieu["daily"])
du_bao

### Giữ lại bản dự báo đã lấy

Gọi API lần nữa có thể nhận được dự báo khác. Lưu nguyên nội dung `r.content` để phân tích lại cùng dữ liệu, kèm cả đơn vị và múi giờ.

Tên file ghi thời điểm tải theo giờ UTC (cách định dạng thời gian học ở Bài 8), nên mỗi lần tải có một bản riêng. File `.meta.json` đi kèm ghi URL đầy đủ và thời điểm tải theo UTC. Trong chương trình thu thập, đặt bước lưu sát bước nhận và kiểm phản hồi.

In [ ]:
tai_luc = pd.Timestamp.now(tz="UTC")
moc = tai_luc.strftime("%Y%m%d-%H%M%S")
tep_json = Path("data/raw") / f"du_bao_{moc}.json"
tep_json.write_bytes(r.content)
tep_json.with_suffix(".meta.json").write_text(
    json.dumps({"url": r.url, "tai_luc_utc": tai_luc.isoformat()},
               ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Đã lưu:", tep_json)

### Phân tích lại từ file

Dùng `tep_json` của lần tải cần phân tích; `json.loads()` chuyển văn bản trong file thành đối tượng Python, tương tự `r.json()` đọc nội dung phản hồi.

Chỉ chạy lại ô dưới nếu muốn dùng cùng đầu vào. Nếu đã mở một phiên notebook mới, gán `tep_json = Path("đường/dẫn/tới/file_đã_lưu.json")` trước khi đọc.

In [ ]:
du_lieu = json.loads(tep_json.read_text(encoding="utf-8"))
du_bao = pd.DataFrame(du_lieu["daily"])

### Bài tập 2: Lọc ngày nóng

`phan_hoi` dưới đây là dữ liệu giả lập. Công thức đổi đơn vị: **°C = (°F − 32) / 1,8**.

**Từ phản hồi này, lấy các ngày có nhiệt độ tối đa từ 25°C trở lên.**

In [ ]:
phan_hoi = {
    "daily_units": {"temperature_2m_max": "°F"},
    "daily": {
        "time": ["2026-10-01", "2026-10-02", "2026-10-03"],
        "temperature_2m_max": [77, 82, 68],
    },
}

In [ ]:
# Bài tập 2: viết mã của bạn ở đây.

## 3. SQL và DuckDB

### Lấy kết quả từ cơ sở dữ liệu

Trong nhiều hệ thống, dữ liệu nằm trong cơ sở dữ liệu. Nếu báo cáo chỉ cần giá trung bình từng khu, ta có thể yêu cầu hệ quản trị tính từ bảng chỗ ở rồi lấy bảng tổng hợp về Python.

**SQL** (Structured Query Language) là ngôn ngữ để diễn đạt yêu cầu đó: chọn hàng, tính theo nhóm, ghép bảng. Ta mô tả kết quả cần lấy; hệ quản trị quyết định cách thực hiện. Các thao tác này đã quen từ Bài 5, nhưng ở đây chúng có thể được thực hiện tại nơi quản lý dữ liệu trước khi trả kết quả về chương trình.

### Dùng DuckDB trong Python

Trong bài này, dữ liệu đang là file và DataFrame trên máy. Ta cần một công cụ thực hiện truy vấn SQL trên các nguồn đó.

**DuckDB** là hệ quản trị cơ sở dữ liệu dành cho phân tích, phù hợp với các phép tính tổng hợp và ghép bảng. Thư viện Python chạy cùng chương trình, không cần máy chủ riêng, và truy vấn được trực tiếp DataFrame hoặc file CSV, Parquet. Chỉ cần đưa bảng kết quả về pandas để xử lý tiếp.

`duckdb.sql(...)` nhận câu truy vấn; `.df()` lấy kết quả dưới dạng DataFrame. Thư viện đã được cài ở ô chuẩn bị đầu notebook.

Bắt đầu với `df` gồm năm chỗ ở giả lập đã đọc ở mục 1 để tính tay và kiểm chứng. Tên biến `df` được dùng làm tên bảng trong truy vấn. Cuối mục, ta sẽ truy vấn trực tiếp file Parquet đã lưu.

Nguồn: [McKinney §6.4](https://wesmckinney.com/book/accessing-data#io_databases), [giới thiệu DuckDB](https://duckdb.org/why_duckdb).

In [ ]:
print(df.to_string(index=False))

### Chọn chỗ ở có giá từ 100 USD/đêm

`FROM` chỉ nguồn bảng, `WHERE` lọc hàng, `SELECT` chọn cột và `ORDER BY` sắp xếp kết quả. `DESC` là thứ tự giảm dần. Hàng `005` có giá thiếu: phép so sánh `gia >= 100` không trả về true nên hàng này không có trong kết quả.

In [ ]:
import duckdb

ket_qua = duckdb.sql("""
    SELECT id, khu, gia
    FROM df
    WHERE gia >= 100
    ORDER BY gia DESC
""").df()
print(ket_qua.to_string(index=False))

### Tính giá trung bình từng khu

`GROUP BY` gom các hàng theo khu. `AVG(gia)` tính trung bình các giá đã biết; `COUNT(*)` đếm hàng và `COUNT(gia)` đếm giá không thiếu. `AS` đặt tên cột kết quả.

Khi DuckDB đọc DataFrame, NaN của pandas thành `NULL`; `AVG` và `COUNT(gia)` bỏ qua `NULL`. Một giá trị NaN thật trong SQL thì không bị bỏ qua.

In [ ]:
theo_khu = duckdb.sql("""
    SELECT khu, AVG(gia) AS trung_binh,
           COUNT(*) AS so_cho_o, COUNT(gia) AS so_gia
    FROM df
    GROUP BY khu
    ORDER BY khu
""").df()
print(theo_khu.to_string(index=False))

In [ ]:
kiem_chung = df.groupby("khu", as_index=False).agg(
    trung_binh=("gia", "mean"), so_cho_o=("gia", "size"), so_gia=("gia", "count")
)
pd.testing.assert_frame_equal(theo_khu, kiem_chung)

### Lọc hàng và lọc nhóm

`WHERE gia >= 150` loại các chỗ ở dưới ngưỡng trước khi tính trung bình. `HAVING AVG(gia) >= 150` giữ các khu có trung bình đạt ngưỡng sau khi tính trên nhóm. Chạy hai ô dưới rồi giải thích vì sao khu B vắng mặt ở cả hai kết quả.

In [ ]:
loc_hang = duckdb.sql("""
    SELECT khu, AVG(gia) AS tb
    FROM df
    WHERE gia >= 150
    GROUP BY khu
""").df()
print(loc_hang.to_string(index=False))

In [ ]:
loc_nhom = duckdb.sql("""
    SELECT khu, AVG(gia) AS tb
    FROM df
    GROUP BY khu
    HAVING AVG(gia) >= 150
""").df()
print(loc_nhom.to_string(index=False))

Khu B vắng mặt vì hai lý do khác nhau. Với `WHERE`, khu B không còn hàng nào sau khi lọc nên không có nhóm B. Với `HAVING`, nhóm B có trung bình 100, dưới ngưỡng 150.

### Bài tập 3: Đếm giá thiếu

Dùng bảng `df` ở mục 1.

**Viết một truy vấn SQL đếm số chỗ ở chưa có giá của từng khu, kể cả khu không thiếu giá.**

In [ ]:
# Bài tập 3: viết truy vấn của bạn ở đây.

### Ghép thêm vùng

Bảng tra cứu `khu_vung` có thông tin cho khu A; `khu_vung.vung` là cột `vung` của bảng `khu_vung`. `LEFT JOIN` giữ các chỗ ở ở bên trái; hai hàng khu B không tìm được vùng nên nhận giá trị thiếu.

In [ ]:
khu_vung = pd.DataFrame({"khu": ["A"], "vung": ["Bắc"]})

In [ ]:
ghep = duckdb.sql("""
    SELECT df.id, df.khu, khu_vung.vung
    FROM df LEFT JOIN khu_vung ON df.khu = khu_vung.khu
    ORDER BY df.id
""").df()
print(ghep.to_string(index=False))

### Truy vấn trực tiếp file Parquet

Bây giờ cần bảng tổng hợp theo loại chỗ ở từ `listings.parquet` đã lưu ở mục 1. Dữ liệu Santiago, snapshot 29/06/2026, giá CLP/đêm.

SQL mô tả phép tính; DuckDB thực hiện trên file Parquet. Câu truy vấn chỉ cần các cột `room_type` và `price`; cách Parquet lưu theo cột cho phép bộ đọc bỏ qua dữ liệu của các cột khác. Không cần đọc toàn bộ file vào DataFrame trước: `.df()` đưa bảng kết quả, mỗi loại chỗ ở một hàng, về pandas.

`COUNT(price)` chỉ đếm giá đã biết. Sau khi chạy, đối chiếu kết quả với pandas trên cùng đầu vào.

Nguồn: [DuckDB đọc Parquet](https://duckdb.org/docs/current/data/parquet/overview).

In [ ]:
ket_qua_that = duckdb.sql("""
SELECT room_type, MEDIAN(price) AS trung_vi,
       COUNT(price) AS so_gia
FROM 'data/processed/listings.parquet'
GROUP BY room_type
ORDER BY trung_vi DESC
""").df()
print(ket_qua_that.to_string(index=False))

In [ ]:
pandas_that = (
    listings.groupby("room_type")["price"]
    .agg(trung_vi="median", so_gia="count")
    .reset_index()
    .sort_values("trung_vi", ascending=False, ignore_index=True)
)
pd.testing.assert_frame_equal(ket_qua_that, pandas_that)
print("Kết quả SQL khớp với pandas.")

Kết quả SQL khớp với `groupby` của pandas. Trung vị cao nhất (Hotel room) chỉ dựa trên 56 giá, Shared room 59 giá: khi xếp hạng theo trung vị, xem kèm số giá.

## Làm việc với AI thì sao?

AI có thể viết mã đọc file, gọi API và truy vấn SQL khi được cung cấp mẫu dữ liệu và quy ước về mã, đơn vị, giá trị thiếu. AI hay bịa tên tham số API và đoán cấu trúc JSON.

Để kiểm chứng, đối chiếu tên tham số với tài liệu của dịch vụ và xem `r.url`; tính tay trên bảng nhỏ trước rồi đối chiếu SQL với pandas trên **cùng đầu vào**. Hai công cụ cho kết quả giống nhau vẫn có thể cùng tính sai yêu cầu.

## Đọc thêm

- [McKinney, chương 6](https://wesmckinney.com/book/accessing-data): §6.1 đọc/ghi văn bản, §6.2 định dạng nhị phân (có Parquet), §6.3 API, §6.4 cơ sở dữ liệu.
- [Apache Parquet: lưu trữ theo cột](https://parquet.apache.org/docs/overview/).
- [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) và [Parquet](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html).
- [Requests: timeout và lỗi HTTP](https://requests.readthedocs.io/en/latest/user/quickstart/) và [Open-Meteo](https://open-meteo.com/en/docs).
- DuckDB: [giới thiệu](https://duckdb.org/why_duckdb), [SQL trên pandas](https://duckdb.org/docs/current/guides/python/sql_on_pandas), [hàm tổng hợp](https://duckdb.org/docs/current/sql/functions/aggregates), [HAVING](https://duckdb.org/docs/current/sql/query_syntax/having).

## 📖 Tự học

Các ví dụ dưới bổ sung thao tác đọc/ghi; không thuộc phần trình bày chính trên lớp.

### Đọc file nén và cột ngày

Tạo bản nén của chính CSV giả lập để thử: pandas nhận diện `.gz` từ tên file. `parse_dates` yêu cầu chuyển cột ngày sang kiểu ngày giờ; sau khi đọc vẫn cần kiểm tra kiểu và nội dung.

In [ ]:
import gzip

with gzip.open("data/raw/cho_o.csv.gz", "wt", encoding="utf-8") as f:
    f.write(csv_text)
nen = pd.read_csv(
    "data/raw/cho_o.csv.gz", dtype={"id": "str", "gia": "float64"},
    keep_default_na=False, na_values={"gia": ["?"]},
)
pd.testing.assert_frame_equal(nen, df)

co_ngay = pd.read_csv(
    StringIO("id,ngay\n001,2026-06-29\n002,2026-06-30\n"),
    dtype={"id": "str"}, parse_dates=["ngay"],
)
print(co_ngay.dtypes)

### Đo kích thước và thời gian đọc file

Đo trên cùng bốn cột của Santiago. Kết quả phụ thuộc dữ liệu, môi trường và bộ nhớ đệm; các số đo trên máy bạn không nhất thiết giống máy khác.

In [ ]:
listings.to_csv("data/processed/listings.csv", index=False)
for ten, doc in [
    ("listings.csv", lambda p: pd.read_csv(p, dtype={"id": "str"})),
    ("listings.parquet", pd.read_parquet),
]:
    tep = Path("data/processed") / ten
    doc(tep)  # Đọc trước một lần để nạp thư viện.
    thoi_gian = []
    for _ in range(5):
        bat_dau = time.perf_counter()
        doc(tep)
        thoi_gian.append(time.perf_counter() - bat_dau)
    print(ten, f"{tep.stat().st_size / 1e6:.2f} MB",
          f"nhanh nhất trong 5 lần: {min(thoi_gian) * 1000:.1f} ms")

### Tổng hợp khi đọc từng khối

Có thể cộng dồn tổng giá và số giá đã biết mà không giữ toàn bộ bảng trong RAM. Muốn tính trung bình chung, chia **tổng giá** cho **tổng số giá đã biết**; trung bình các trung bình của từng khối có thể sai khi số giá ở các khối khác nhau.

In [ ]:
tong_gia = 0.0
so_gia = 0
for khoi in pd.read_csv("data/raw/santiago_2026-06-29.csv",
                        usecols=["price"], chunksize=5000):
    tong_gia += khoi["price"].sum()
    so_gia += khoi["price"].count()
trung_binh = tong_gia / so_gia
print("Giá trung bình (CLP/đêm):", trung_binh)
assert abs(trung_binh - listings["price"].mean()) < 1e-8

### API key và giới hạn truy cập

Endpoint Open-Meteo trong ví dụ không cần key. Với dịch vụ yêu cầu key, đọc từ biến môi trường hoặc Colab Secrets; không ghi key trong notebook hay đưa vào prompt.

Khi gọi nhiều yêu cầu, tuân thủ giới hạn của dịch vụ và hướng dẫn khi nhận mã 429. Không có một khoảng nghỉ cố định phù hợp với mọi API.

---

## Tóm tắt bài học

- Khai báo kiểu và quy ước giá trị thiếu khi chúng ảnh hưởng đến mã hay phép tính.
- Giữ file gốc; dùng Parquet khi cần lưu bảng đã xử lý, giữ kiểu và đọc lại các cột cần dùng.
- Dùng API để chương trình lấy dữ liệu từ dịch vụ theo yêu cầu; kiểm tra cấu trúc và đơn vị trước khi tạo bảng.
- SQL diễn đạt truy vấn; DuckDB thực hiện trên file hoặc DataFrame và đưa bảng kết quả về pandas.
- Dùng bảng nhỏ để tính tay, rồi kiểm chứng trên snapshot đã lưu.

Chạy **Restart & Run all** để kiểm tra notebook từ đầu. Các ô thu thập API sẽ tải một dự báo mới; muốn phân tích lại cùng bản, chỉ chạy các ô đọc file `tep_json` đã lưu.

**Bài sau:** xử lý dữ liệu chuỗi bằng `str` và biểu thức chính quy (regex).